# 00 — Data inspection: LLD-MMRI-MedSAM2 mirror

**Purpose.** Establish ground truth about the downloaded dataset *before* any pipeline code is written. This notebook makes **no assumptions** about the annotation JSON schema, the meaning of the middle digit in filenames, or whether volumes are full-abdomen or pre-cropped ROIs. It discovers each of these and writes the findings to `data_report.md`.

**How to use.** Run every cell top to bottom on your Mac (CPU only, no GPU needed). Then: (1) read `data_report.md`; (2) give it to Claude Code alongside `CLAUDE_CODE_PROMPT.md`; (3) paste the printed output of cells 3–8 back into the Claude chat if asked.

Runtime: ~2–5 minutes. Requires: `nibabel`, `numpy` (`pip install nibabel numpy`).

In [1]:
# Cell 1 — paths and file counts
from pathlib import Path
import json, re, random, collections

random.seed(42)

ROOT = Path("LLD-MMRI-MedSAM2")  # adjust if your folder sits elsewhere
IMG_DIR = ROOT / "images"
LAB_DIR = ROOT / "labels"
ANN_PATH = ROOT / "LLD_MMRI_Annotation.json"

assert ROOT.exists(), f"Dataset root not found at {ROOT.resolve()} — fix ROOT."

imgs = sorted(IMG_DIR.glob("*.nii.gz"))
labs = sorted(LAB_DIR.glob("*.nii.gz"))
print(f"images/: {len(imgs)} .nii.gz files")
print(f"labels/: {len(labs)} .nii.gz files")
print(f"annotation json exists: {ANN_PATH.exists()} ({ANN_PATH.stat().st_size/1e6:.1f} MB)" if ANN_PATH.exists() else "annotation json MISSING")

# Expected: 3984 images (498 patients x 8 phases). Assert but do not crash — record instead.
findings = {}
findings["n_images"] = len(imgs)
findings["n_labels"] = len(labs)
if len(imgs) != 3984:
    print(f"WARNING: expected 3984 images, found {len(imgs)}")
if len(imgs) != len(labs):
    print(f"WARNING: image/label count mismatch: {len(imgs)} vs {len(labs)}")

images/: 3984 .nii.gz files
labels/: 3984 .nii.gz files
annotation json exists: True (19.0 MB)


In [2]:
# Cell 2 — print the dataset's own README verbatim (provenance documentation)
readme = ROOT / "README.md"
if readme.exists():
    txt = readme.read_text(errors="replace")
    print(txt)
    findings["dataset_readme"] = txt
else:
    print("No README.md found inside dataset folder.")
    findings["dataset_readme"] = "(absent)"

---
task_categories:
- image-segmentation
language:
- en
tags:
- medical
size_categories:
- 1K<n<10K
---
# LLD-MMRI-MedSAM2 Dataset

<div align="center">
 <table align="center">
   <tr>
     <td><a href="https://arxiv.org/abs/2504.03600" target="_blank"><img src="https://img.shields.io/badge/arXiv-Paper-FF6B6B?style=for-the-badge&logo=arxiv&logoColor=white" alt="Paper"></a></td>
     <td><a href="https://medsam2.github.io/" target="_blank"><img src="https://img.shields.io/badge/Project-Page-4285F4?style=for-the-badge&logoColor=white" alt="Project"></a></td>
     <td><a href="https://github.com/bowang-lab/MedSAM2" target="_blank"><img src="https://img.shields.io/badge/GitHub-Code-181717?style=for-the-badge&logo=github&logoColor=white" alt="Code"></a></td>
     <td><a href="https://huggingface.co/wanglab/MedSAM2" target="_blank"><img src="https://img.shields.io/badge/HuggingFace-Model-FFBF00?style=for-the-badge&logo=huggingface&logoColor=white" alt="HuggingFace Model"></a></td>
   </tr>


In [3]:
# Cell 3 — annotation JSON: schema EXPLORER (assumes nothing)
with open(ANN_PATH) as f:
    ann = json.load(f)

print("Top-level type:", type(ann).__name__)

def explore(node, prefix="", depth=0, max_depth=4, key_counter=None, samples=None):
    """Recursively record key paths and one example value per path."""
    if key_counter is None: key_counter = collections.Counter()
    if samples is None: samples = {}
    if depth > max_depth: return key_counter, samples
    if isinstance(node, dict):
        for k, v in node.items():
            path = f"{prefix}.{k}" if prefix else k
            key_counter[path] += 1
            if path not in samples and not isinstance(v, (dict, list)):
                samples[path] = repr(v)[:120]
            explore(v, path, depth+1, max_depth, key_counter, samples)
    elif isinstance(node, list) and node:
        key_counter[f"{prefix}[] (len={len(node)})"] += 1
        explore(node[0], prefix + "[]", depth+1, max_depth, key_counter, samples)
    return key_counter, samples

keys, samples = explore(ann)
print("\n--- Key paths (path : occurrences) ---")
for k, c in sorted(keys.items()):
    print(f"{k} : {c}")
print("\n--- Example scalar values ---")
for k, v in list(samples.items())[:40]:
    print(f"{k} = {v}")
findings["json_key_paths"] = dict(keys)

Top-level type: dict

--- Key paths (path : occurrences) ---
Annotation_info : 1
Annotation_info.MR-391135 : 1
Annotation_info.MR-391135[] (len=8) : 1
Annotation_info.MR-391135[].annotation : 1
Annotation_info.MR-391135[].annotation.lesion : 1
Annotation_info.MR-391135[].annotation.num_targets : 1
Annotation_info.MR-391135[].origin : 1
Annotation_info.MR-391135[].origin[] (len=3) : 1
Annotation_info.MR-391135[].phase : 1
Annotation_info.MR-391135[].pixel_spacing : 1
Annotation_info.MR-391135[].pixel_spacing[] (len=2) : 1
Annotation_info.MR-391135[].seriesUID : 1
Annotation_info.MR-391135[].slice_spacing : 1
Annotation_info.MR-391135[].slice_thickness : 1
Annotation_info.MR-391135[].studyUID : 1
Annotation_info.MR-398189 : 1
Annotation_info.MR-398189[] (len=8) : 1
Annotation_info.MR-398189[].annotation : 1
Annotation_info.MR-398189[].annotation.lesion : 1
Annotation_info.MR-398189[].annotation.num_targets : 1
Annotation_info.MR-398189[].origin : 1
Annotation_info.MR-398189[].origin[] (l

In [4]:
# Cell 4 — hunt for the fields that matter: class labels and train/val/test split
IMPORTANT = ("class", "category", "label", "lesion", "split", "train", "val", "test", "diagnos", "type", "phase", "malig", "benign", "bbox", "box")

def hunt(node, prefix="", hits=None, depth=0, max_depth=6):
    if hits is None: hits = []
    if depth > max_depth: return hits
    if isinstance(node, dict):
        for k, v in node.items():
            path = f"{prefix}.{k}" if prefix else k
            if any(t in k.lower() for t in IMPORTANT):
                preview = repr(v)[:200]
                hits.append((path, preview))
            hunt(v, path, hits, depth+1, max_depth)
    elif isinstance(node, list) and node:
        hunt(node[0], prefix + "[]", hits, depth+1, max_depth)
    return hits

hits = hunt(ann)
print(f"{len(hits)} candidate fields containing class/split/lesion keywords:\n")
for path, preview in hits[:60]:
    print(f"{path}\n    -> {preview}\n")
findings["candidate_fields"] = [p for p, _ in hits]

1995 candidate fields containing class/split/lesion keywords:

Annotation_info.MR-398189[].phase
    -> 'T2WI'

Annotation_info.MR-398189[].annotation.lesion
    -> {'0': {'category': 1, 'bbox': {'2D_box': [{'slice_idx': 10, 'x_min': 132.45, 'y_min': 290.38, 'x_max': 170.47, 'y_max': 327.58002, 'area': 1414.3447604000003}, {'slice_idx': 11, 'x_min': 131.64, 'y_mi

Annotation_info.MR-398189[].annotation.lesion.0.category
    -> 1

Annotation_info.MR-398189[].annotation.lesion.0.bbox
    -> {'2D_box': [{'slice_idx': 10, 'x_min': 132.45, 'y_min': 290.38, 'x_max': 170.47, 'y_max': 327.58002, 'area': 1414.3447604000003}, {'slice_idx': 11, 'x_min': 131.64, 'y_min': 281.48, 'x_max': 181.79001

Annotation_info.MR-410252[].phase
    -> 'T2WI'

Annotation_info.MR-410252[].annotation.lesion
    -> {'0': {'category': 1, 'bbox': {'2D_box': [{'slice_idx': 5, 'x_min': 205.25, 'y_min': 190.08, 'x_max': 283.71, 'y_max': 245.89, 'area': 4378.8525999999965}, {'slice_idx': 6, 'x_min': 200.39, 'y_min': 1



In [5]:
# Cell 5 — filename anatomy: phases, middle digit, patient IDs, hyphen normalisation
PATTERN = re.compile(r"^(MR-?\d+)_(\d+)_([A-Za-z0-9+\-]+?)(_0000)?\.nii\.gz$")

parsed, unparsed = [], []
for p in imgs:
    m = PATTERN.match(p.name)
    (parsed if m else unparsed).append((p.name, m.groups() if m else None))

print(f"parsed: {len(parsed)}, unparsed: {len(unparsed)}")
if unparsed:
    print("UNPARSED examples (fix regex before proceeding):")
    for n, _ in unparsed[:10]: print("   ", n)

raw_ids   = sorted({g[0] for _, g in parsed})
norm_ids  = sorted({g[0].replace("MR-", "MR") for _, g in parsed})
digits    = collections.Counter(g[1] for _, g in parsed)
phases    = collections.Counter(g[2] for _, g in parsed)
suffixes  = collections.Counter(g[3] or "(none)" for _, g in parsed)

print(f"\nraw patient IDs: {len(raw_ids)}  |  hyphen-normalised: {len(norm_ids)}")
if len(raw_ids) != len(norm_ids):
    print("CRITICAL: hyphen normalisation MERGES ids — MR-X and MRX refer to the same file family. "
          "Investigate before normalising; a wrong merge corrupts the patient-level split.")
hyphenated = [i for i in raw_ids if i.startswith("MR-")]
print(f"hyphenated IDs: {len(hyphenated)} e.g. {hyphenated[:5]}")
print(f"\nmiddle-digit distribution: {dict(sorted(digits.items()))}")
print(f"phase names: {dict(sorted(phases.items()))}")
print(f"filename suffixes: {dict(suffixes)}")

# per-patient completeness: 8 phases each?
per_patient = collections.Counter(g[0] for _, g in parsed)
sizes = collections.Counter(per_patient.values())
print(f"\nfiles per patient-ID -> count of patients: {dict(sorted(sizes.items()))}")
# does the middle digit vary WITHIN a patient? If constant per patient, it is plausibly the class code.
digit_per_patient = collections.defaultdict(set)
for _, g in parsed: digit_per_patient[g[0]].add(g[1])
varying = [pid for pid, ds in digit_per_patient.items() if len(ds) > 1]
print(f"patients whose middle digit VARIES across files: {len(varying)} "
      f"({'digit is NOT a per-patient constant -> not simply the class' if varying else 'digit constant per patient -> plausibly class code, VERIFY against JSON'})")
findings.update(n_patients_raw=len(raw_ids), n_patients_norm=len(norm_ids),
                middle_digit_dist=dict(sorted(digits.items())), phases=sorted(phases),
                digit_constant_per_patient=(len(varying) == 0))

parsed: 3984, unparsed: 0

raw patient IDs: 498  |  hyphen-normalised: 498
hyphenated IDs: 16 e.g. ['MR-391135', 'MR-398189', 'MR-398374', 'MR-400851', 'MR-401097']

middle-digit distribution: {'0': 632, '1': 464, '2': 432, '3': 408, '4': 424, '5': 368, '6': 1256}
phase names: {'C+A': 498, 'C+Delay': 498, 'C+V': 498, 'C-pre': 498, 'DWI': 498, 'InPhase': 498, 'OutPhase': 498, 'T2WI': 498}
filename suffixes: {'_0000': 3984}

files per patient-ID -> count of patients: {8: 498}
patients whose middle digit VARIES across files: 0 (digit constant per patient -> plausibly class code, VERIFY against JSON)


In [6]:
# Cell 6 — label file pairing: discover the naming convention empirically
lab_names = {p.name for p in labs}
sample_imgs = random.sample(imgs, 8)
print("pairing check on 8 random images:")
pair_rule = None
for p in sample_imgs:
    candidates = [p.name, p.name.replace("_0000", "")]
    found = next((c for c in candidates if c in lab_names), None)
    print(f"  {p.name}  ->  label: {found}")
    if found: pair_rule = "identical" if found == p.name else "strip _0000"
print(f"\ninferred pairing rule: {pair_rule}")
findings["label_pairing_rule"] = pair_rule

pairing check on 8 random images:
  MR211043_0_C-pre_0000.nii.gz  ->  label: MR211043_0_C-pre.nii.gz
  MR119513_2_C+A_0000.nii.gz  ->  label: MR119513_2_C+A.nii.gz
  MR-462969_5_OutPhase_0000.nii.gz  ->  label: MR-462969_5_OutPhase.nii.gz
  MR24890_6_InPhase_0000.nii.gz  ->  label: MR24890_6_InPhase.nii.gz
  MR160269_4_OutPhase_0000.nii.gz  ->  label: MR160269_4_OutPhase.nii.gz
  MR147038_6_C-pre_0000.nii.gz  ->  label: MR147038_6_C-pre.nii.gz
  MR14362_6_C+V_0000.nii.gz  ->  label: MR14362_6_C+V.nii.gz
  MR125176_4_C-pre_0000.nii.gz  ->  label: MR125176_4_C-pre.nii.gz

inferred pairing rule: strip _0000


In [7]:
# Cell 7 — geometry: full volumes or pre-cropped ROIs? (decides the whole compute plan)
import numpy as np, nibabel as nib

def label_for(img_path):
    for cand in (img_path.name, img_path.name.replace("_0000", "")):
        q = LAB_DIR / cand
        if q.exists(): return q
    return None

# one sample per phase name
by_phase = {}
for p in imgs:
    m = PATTERN.match(p.name)
    if m and m.group(3) not in by_phase:
        by_phase[m.group(3)] = p

rows = []
for phase, p in sorted(by_phase.items()):
    img = nib.load(str(p)); shape = img.shape; zoom = tuple(round(z, 2) for z in img.header.get_zooms()[:3])
    lp = label_for(p)
    if lp is not None:
        arr = np.asanyarray(nib.load(str(lp)).dataobj)
        uniq = np.unique(arr); fg = int((arr > 0).sum())
        lab_info = f"mask shape={arr.shape} unique={uniq[:6]} fg_voxels={fg}"
        same_grid = (arr.shape == shape)
    else:
        lab_info, same_grid = "NO LABEL FOUND", None
    rows.append((phase, p.name, shape, zoom, same_grid))
    print(f"{phase:10s} {p.name}\n    image shape={shape} spacing={zoom} | {lab_info} | grids match: {same_grid}")

vox = [np.prod(s) for _, _, s, _, _ in rows]
verdict = ("PRE-CROPPED ROIs (small volumes) -> training is cheap; free GPU tiers ample"
           if max(vox) < 2_000_000 else
           "FULL VOLUMES -> crop lesion ROIs locally on CPU using masks BEFORE any GPU work")
print(f"\nmax voxels in sample: {max(vox):,} -> {verdict}")
findings["geometry_verdict"] = verdict
findings["sample_shapes"] = [(r[0], list(r[2]), list(r[3])) for r in rows]

ModuleNotFoundError: No module named 'nibabel'

In [8]:
# Cell 8 — write data_report.md (the file Claude Code must read before building anything)
lines = ["# LLD-MMRI-MedSAM2 — data inspection report",
         "",
         f"- images: {findings['n_images']}  |  labels: {findings['n_labels']}",
         f"- patients (raw IDs): {findings.get('n_patients_raw')}  |  after hyphen-normalisation: {findings.get('n_patients_norm')}",
         f"- phases found: {findings.get('phases')}",
         f"- middle-digit distribution: {findings.get('middle_digit_dist')}",
         f"- middle digit constant per patient: {findings.get('digit_constant_per_patient')} (if True, plausibly class code — MUST be verified against JSON before use)",
         f"- label pairing rule: {findings.get('label_pairing_rule')}",
         f"- geometry verdict: {findings.get('geometry_verdict')}",
         "",
         "## Sample shapes (phase, shape, spacing mm)",
         *[f"- {p}: {s} @ {z}" for p, s, z in findings.get('sample_shapes', [])],
         "",
         "## Annotation JSON key paths",
         *[f"- {k} ({c})" for k, c in sorted(findings.get('json_key_paths', {}).items())],
         "",
         "## Candidate class/split fields (verify semantics before use)",
         *[f"- {p}" for p in findings.get('candidate_fields', [])[:40]],
         "",
         "## Dataset's own README (verbatim)",
         "```", findings.get('dataset_readme', '(absent)'), "```"]
Path("data_report.md").write_text("\n".join(str(x) for x in lines))
print("Wrote data_report.md — read it, then hand it to Claude Code.")

Wrote data_report.md — read it, then hand it to Claude Code.


## Next steps

1. Read `data_report.md` yourself — especially the **geometry verdict**, the **middle-digit** finding, and the **hyphen-ID** warning if it fired.
2. Start Claude Code in the repo root and point it at `CLAUDE_CODE_PROMPT.md`. Its first mandated action is to read `data_report.md`.
3. If anything above printed a WARNING or CRITICAL line, resolve it (with Claude Code or in chat) **before** the manifest is built. A wrong assumption at this stage silently corrupts every result downstream.